In [55]:
import pandas as pd
import numpy as np 
from sklearn.impute import KNNImputer
import pandera as pa
from pandera import Column, Check, DataFrameSchema

# loading raw data
df_raw = pd.read_csv(r'C:\Users\User\OneDrive\Desktop\Advance_Regression\train.csv')
df = df_raw.copy()


# Phase One: Securing Input Fidelity

In [13]:
print('Missing Values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing Values:
LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64


In [54]:
# calculate missingness proportion per feature
missing_pct = df.isnull().mean()

# dropping rows where missingness is < 5%
cols_drop_rows = missing_pct[(missing_pct > 0) & (missing_pct < 0.05)].index
df = df.dropna(subset = cols_drop_rows)

# skewed numeric imputation using global median 5% - 20% missingness
cols_median = missing_pct[(missing_pct >= 0.05) & (missing_pct <= 0.20) & (df.dtypes != 'object')].index
df[cols_median] = df[cols_median].fillna(df[cols_median].median())

# categorical imputation 5 - 20% missingness
df['GarageFinish'] = df.groupby('Neighborhood')['GarageFinish'].transform(lambda x: x.fillna(x.mode()[0] if not x.empty else 'Missing'))

#  complex numerical missingness > 20% 
knn_cols = ['LotFrontage', 'MasVnrArea']
knn_imp = KNNImputer(n_neighbors = 5)
df[knn_cols] = knn_imp.fit_transform(df[knn_cols])

# winsorization
def winsorized_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return np.clip(series, a_min = lower_bound, a_max = upper_bound)
df['GrLivArea_Winsorized'] = winsorize_column(df['GrLivArea'])
df['SalePrice_Winsorized'] = winsorize_column(df['SalePrice'])


# Phase Two: Vectorized Computation Engine

In [53]:
# Vectorized Feature Transformations (No Procedural Loops)
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['IsRemodeled'] = np.where(df['YearBuilt'] != df['YearRemodAdd'], 1, 0)

# Hot-One Encoding (One-Hot Encoding) via Pandas
df = pd.get_dummies(df, columns=['Street', 'Alley'], drop_first=True)

# Phase Three: Structural 

In [48]:
# runtime schema contract enforcement using pandera
output_schema = DataFrameSchema({
    "TotalSF": Column(int, Check.ge(0), nullable=False),
    "HouseAge": Column(int, Check.ge(0), nullable=False),
    "IsRemodeled": Column(int, Check.isin([0, 1]), nullable=False),
    "GrLivArea_Winsorized": Column(float, Check.ge(0), nullable=False),
})

# Validate the transformed dataset against structural contract
validated_df = output_schema.validate(df)


# output: save to high fidelity store
validated_df.to_csv('feature_store_production.csv', index = False)
print('Pipeline execution complete. Data validated and saved to Feature Store')

Pipeline execution complete. Data validated and saved to Feature Store
